In [ ]:
# @title Install required dependencies
!pip install -q "crewai[tools]>=1.8.0" python-dotenv ddgs pydantic pandas pypdf


In [ ]:
import os
from pydantic import BaseModel
from google.colab import userdata
from crewai import Agent, Task, Crew, LLM
from crewai.flow.flow import Flow, and_, listen, start

SAMPLE_ARTICLE = """
Generative AI in 2026 is moving from chat interfaces directly into background agent workflows.
Instead of prompting an LLM for single answers, companies are implementing autonomous multi-agent
orchestrations where specialist agents validate data, query internal APIs, and execute end-to-end
business processes. The biggest shift is reliability: adding structured schema validations and human
approval gates turns experimental prototypes into dependable production systems.
"""

class RepurposingState(BaseModel):
    article: str = SAMPLE_ARTICLE
    linkedin_post: str = ""
    twitter_thread: str = ""
    newsletter_snippet: str = ""

try:
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    llm = LLM(
        model="google/gemini-3.1-flash-lite-preview",
        api_key=userdata.get("GEMINI_API_KEY"),
        temperature=0.6,
        max_tokens=1500,
    )

    class ContentRepurposingFlow(Flow[RepurposingState]):
        @start()
        def load_source_article(self):
            print("Analyzing source article for multi-channel distribution...")

        @listen(load_source_article)
        def create_linkedin(self):
            writer = Agent(
                role="LinkedIn Content Creator",
                goal="Draft an engaging, professional LinkedIn thought leadership post",
                backstory="Growth strategist creating hook-driven professional content.",
                llm=llm,
                verbose=False,
            )
            task = Task(
                description=f"Repurpose into a high-engagement LinkedIn post with hook, key points, and discussion question:\n{self.state.article}",
                expected_output="Formatted LinkedIn post with hashtags.",
                agent=writer,
            )
            self.state.linkedin_post = Crew(agents=[writer], tasks=[task]).kickoff().raw

        @listen(load_source_article)
        def create_twitter(self):
            writer = Agent(
                role="X/Twitter Strategist",
                goal="Draft a punchy 4-tweet thread summarizing key ideas",
                backstory="Social media creator skilled in concise takeaways and strong hooks.",
                llm=llm,
                verbose=False,
            )
            task = Task(
                description=f"Repurpose into a numbered 4-tweet thread from this text:\n{self.state.article}",
                expected_output="A 4-tweet thread separated by tweet numbers (1/4 to 4/4).",
                agent=writer,
            )
            self.state.twitter_thread = Crew(agents=[writer], tasks=[task]).kickoff().raw

        @listen(load_source_article)
        def create_newsletter(self):
            writer = Agent(
                role="Newsletter Editor",
                goal="Draft a weekly newsletter section with TL;DR and actionable tips",
                backstory="Curator of a popular tech newsletter with 50k+ subscribers.",
                llm=llm,
                verbose=False,
            )
            task = Task(
                description=f"Repurpose into a newsletter section featuring TL;DR, Core Takeaway, and Action Item:\n{self.state.article}",
                expected_output="Structured newsletter section in Markdown.",
                agent=writer,
            )
            self.state.newsletter_snippet = Crew(agents=[writer], tasks=[task]).kickoff().raw

        @listen(and_(create_linkedin, create_twitter, create_newsletter))
        def finalize_pack(self):
            print("\n=== Repurposed Content Pack Generated Successfully ===")

    flow = ContentRepurposingFlow()
    flow.kickoff()

    print("\n[1] LinkedIn Post:\n", flow.state.linkedin_post)
    print("\n[2] Twitter/X Thread:\n", flow.state.twitter_thread)
    print("\n[3] Newsletter:\n", flow.state.newsletter_snippet)

except Exception as e:
    print("Error:", e)
